**FLAML (AutoML)**

In [130]:
!pip install flaml

In [131]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from flaml import AutoML

In [132]:
df = pd.read_csv(r"/content/streaming_churn_dataset_BALANCED.csv")

In [133]:
X = df.drop(columns=["user_id", "churn"])

y = df["churn"]

In [134]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [135]:
automl = AutoML()

automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",
    metric="accuracy",
    estimator_list=[
        "rf",          # Random Forest
        "extra_tree",  # Extra Trees
        "xgboost",     # XGBoost
        "lgbm",        # LightGBM
        "xgb_limitdepth", # XGBoost with limited tree depth

    ],
    time_budget=300,
    seed=42
)

[flaml.automl.logger: 07-08 14:39:34] {2375} INFO - task = classification
[flaml.automl.logger: 07-08 14:39:34] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 07-08 14:39:34] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 07-08 14:39:34] {2606} INFO - List of ML learners in AutoML Run: ['rf', 'extra_tree', 'xgboost', 'lgbm', 'xgb_limitdepth']
[flaml.automl.logger: 07-08 14:39:34] {2911} INFO - iteration 0, current learner rf
[flaml.automl.logger: 07-08 14:39:35] {3046} INFO - Estimated sufficient time budget=3461s. Estimated necessary time budget=12s.
[flaml.automl.logger: 07-08 14:39:35] {3097} INFO -  at 0.4s,	estimator rf's best error=1.0525e-01,	best estimator rf's best error=1.0525e-01
[flaml.automl.logger: 07-08 14:39:35] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 07-08 14:39:36] {3097} INFO -  at 1.9s,	estimator lgbm's best error=1.0850e-01,	best estimator rf's best error=1.0525e-01
[flaml.automl.logger: 07-08 14:39:

In [136]:
print("Best Algorithm:", automl.best_estimator)
print("\nBest Hyperparameters:")
print(automl.best_config)

Best Algorithm: extra_tree

Best Hyperparameters:
{'n_estimators': 9, 'max_features': np.float64(0.5639507429896701), 'max_leaves': 10, 'criterion': np.str_('gini')}


In [137]:
y_pred = automl.predict(X_test)

In [138]:
print("Accuracy :", round(accuracy_score(y_test, y_pred)*100,2),"%")
print("Precision:", round(precision_score(y_test, y_pred)*100,2),"%")
print("Recall   :", round(recall_score(y_test, y_pred)*100,2),"%")
print("F1 Score :", round(f1_score(y_test, y_pred)*100,2),"%")

Accuracy : 91.1 %
Precision: 89.6 %
Recall   : 93.0 %
F1 Score : 91.27 %


In [139]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[446  54]
 [ 35 465]]


**1. XGBoost**

In [157]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

# Prediction
y_pred_xgb = xgb_model.predict(X_test)


print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_xgb))


Confusion Matrix
[[444  56]
 [ 32 468]]


**2. Random Forest**

In [158]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)


print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))


Confusion Matrix
[[446  54]
 [ 34 466]]


**3. Extra Trees**

In [159]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_train, y_train)

y_pred_et = et_model.predict(X_test)


print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_et))


Confusion Matrix
[[448  52]
 [ 40 460]]


**4. XGBoost LimitDepth**

In [160]:
from xgboost import XGBClassifier

xgb_limit_model = XGBClassifier(
    max_depth=3,
    random_state=42,
    eval_metric="logloss"
)

xgb_limit_model.fit(X_train, y_train)

y_pred_xgb_limit = xgb_limit_model.predict(X_test)
print("\nConfusion Matrix")
cm = confusion_matrix(y_test, y_pred_xgb_limit)

print(cm)


Confusion Matrix
[[448  52]
 [ 31 469]]


**5. LightGBM**

In [161]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(random_state=42)

lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_lgbm)

print("\nConfusion Matrix")
print(cm)


Confusion Matrix
[[449  51]
 [ 36 464]]


**Comparison Table**

In [162]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Algorithm": [
        "Extra Trees",
        "Random Forest",
        "XGBoost",
        "XGBoost LimitDepth",
        "LightGBM"
    ],

    "Accuracy": [
        accuracy_score(y_test, y_pred_et),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_xgb_limit),
        accuracy_score(y_test, y_pred_lgbm)
    ],

    "Precision": [
        precision_score(y_test, y_pred_et),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb_limit),
        precision_score(y_test, y_pred_lgbm)
    ],

    "Recall": [
        recall_score(y_test, y_pred_et),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb_limit),
        recall_score(y_test, y_pred_lgbm)
    ],

    "F1 Score": [
        f1_score(y_test, y_pred_et),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb_limit),
        f1_score(y_test, y_pred_lgbm)
    ]
})

comparison.iloc[:, 1:] = comparison.iloc[:, 1:] * 100
comparison = comparison.round(2)

comparison = comparison.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print(comparison)

            Algorithm  Accuracy  Precision  Recall  F1 Score
0  XGBoost LimitDepth      91.7      90.02    93.8     91.87
1            LightGBM      91.3      90.10    92.8     91.43
2             XGBoost      91.2      89.31    93.6     91.41
3       Random Forest      91.2      89.62    93.2     91.37
4         Extra Trees      90.8      89.84    92.0     90.91
